In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("statsmodelsOther.ipynb")

## Lecture Section

In this lecture, we will cover more of the `statsmodels` library.
We will cover:
* MICE imputation
* Generalized Linear Models (GLMs)
* Weighted Least Squares (WLS)


### MICE

MICE stands for "multiple imputation with chained equations" - it's one method of 'fixing' missing data, also called `imputation`. It doesn't work well when your data is categorical, so we are going to use the `sleepstudy` data. It contains reaction times of a group of people who were sleep-deprived during a research study.

In [ ]:
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.imputation.mice import MICEData
from statsmodels.datasets import get_rdataset
import pandas as pd

# grab the data!
sleep = get_rdataset("sleepstudy", "lme4").data
sleep.head()


df = pd.DataFrame(sleep)

# count missing values per column
missing_data = df.isnull().sum()
print(missing_data)

There aren't any missing data, so I will introduce some.

In [ ]:
# remove the cetegorical variable - could be re-added later
sleep_clean = sleep.drop(columns=["Subject"])

# this is just introducing some missing data...
sleep_missing = sleep_clean.copy()
sleep_missing.loc[0:7, "Reaction"] = np.nan

# count missing values per column
df = pd.DataFrame(sleep_missing)
missing_data = df.isnull().sum()
print(missing_data)

We can work with that! We have 8 missing values in our `Reaction` column.

In [ ]:
# load in the MICE function and give it our data
mice_data = MICEData(sleep_missing)
print("Imputed data preview:")
print(mice_data.data.head(10))
print()

# count missing values per column
df = pd.DataFrame(mice_data.data)
missing_data = df.isnull().sum()
print(missing_data)
print()

# run our regression model on the imputed data!
model = smf.ols("Reaction ~ Days", data=mice_data.data).fit()
print(model.summary())

N ow we don't have any missing columns, and we can run our model! We couldn't run it before - you can try to run it with the `missing_data` data, but it will give you an error.

MICE is one of the `easiest` imputation methods - but there are others.

Other methods: https://pyquesthub.com/a-comprehensive-guide-to-data-imputation-techniques-in-python-for-accurate-data-analysis
MICE: https://www.statsmodels.org/stable/imputation.html

### Generalized Linear Models

Linear regression that isn't so linear, or doesn't nicely fit the assumptions of regular OLS, can still be modeling with regression. These models generalize into GLMs, or Generalized Linear Models. These are highly customizable - you can do regular linear regression, logistic regression, or something else. You can model if your outcome/response is binary (Yes/No; 0/1), skewed, etc. GLMs allow the response variable to come from other distributions in the exponential family, too. You can read more about them here:

https://www.statology.org/a-beginners-guide-to-generalized-linear-models-glms/

In this example, we fit data with a gamma distribution.

In [ ]:
import statsmodels.api as sm

data = sm.datasets.scotland.load()
data.exog = sm.add_constant(data.exog)
glm_gamma = sm.GLM(data.endog, data.exog, family=sm.families.Gamma(sm.families.links.Log()))
glm_results = glm_gamma.fit()
print(glm_results.summary())

Here is an example with a binomial response variable - NABOVE & NBELOW.

In [ ]:
data = sm.datasets.star98.load()
data.exog = sm.add_constant(data.exog, prepend=False)
print(sm.datasets.star98.NOTE)
print(data.endog.head())

In [ ]:
glm_binom = sm.GLM(data.endog, data.exog, family=sm.families.Binomial())
res = glm_binom.fit()
print(res.summary())

Not that the Y and x variables are backwards, Y is `endog` and x is `exog`.

Below is helpful data for both of our Y-variable categories.

You can plot for assumptions and more by reading the documentation provided by `statsmodels`.
https://www.statsmodels.org/stable/examples/notebooks/generated/glm.html

In [ ]:
print('Total number of trials:',  data.endog.iloc[:, 0].sum())
print('Parameters: ', res.params)
print('T-values: ', res.tvalues)

### Weighted Least Squares

When doing linear regression, we have several assumptions. One of them is `homoscedasticity` - the residuals are distributed with equal variance. When this is violated, our residuals show `heteroscedasticity`.

We are going to look at some data we looked at for M4A4. One of the questions asked you to report the violations of the assumptions for the `penguins` dataset.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm

penguins = sns.load_dataset('penguins').dropna()
model = smf.ols("body_mass_g ~ flipper_length_mm", data=penguins).fit() # SOLUTION

fitted = model.fittedvalues
residuals = model.resid
standardized_resid = (residuals - residuals.mean()) / residuals.std()

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

# Scatterplot of x vs y
axes[0].scatter(penguins['flipper_length_mm'], penguins['body_mass_g'], alpha=0.5, color='steelblue')
axes[0].set_title('Flipper Length vs. Body Mass')
axes[0].set_xlabel('Flipper Length (mm)')
axes[0].set_ylabel('Body Mass (g)')

# Residuals vs. Fitted
lowess_resid = sm.nonparametric.lowess(residuals, fitted, frac=0.6)
axes[1].scatter(fitted, residuals, alpha=0.5, color='steelblue')
axes[1].plot(lowess_resid[:, 0], lowess_resid[:, 1], color='red', linewidth=1.5)
axes[1].axhline(0, color='gray', linestyle='--')
axes[1].set_title('Residuals vs. Fitted')
axes[1].set_xlabel('Fitted Values')
axes[1].set_ylabel('Residuals')

# Q-Q plot
sm.qqplot(residuals, line='s', ax=axes[2], alpha=0.5)
axes[2].set_title('Q-Q Plot')

# Scale-Location
sqrt_abs = np.sqrt(np.abs(standardized_resid))
lowess_sl = sm.nonparametric.lowess(sqrt_abs, fitted, frac=0.6)
axes[3].scatter(fitted, sqrt_abs, alpha=0.5, color='steelblue')
axes[3].plot(lowess_sl[:, 0], lowess_sl[:, 1], color='red', linewidth=1.5)
axes[3].set_title('Scale-Location')
axes[3].set_xlabel('Fitted Values')
axes[3].set_ylabel('√|Standardized Residuals|')

plt.tight_layout()
plt.show()

Technically, our data is not linear due to a species effect. Separating by species would leave us with homoscedastic residuals, but if we keep them together, we can see that there is some homoscedasticity. We see this in the `Residuals vs Fitted` plot and the `Scale Location` plot.


One way of dealing with this is with weighted least squares regression, where we weight observations so that those with a small error variance are given more weight since they provide better information to our model.

First, we will do a regular linear regression to compare outputs.

In [ ]:
model = smf.ols("body_mass_g ~ flipper_length_mm", data=penguins).fit() #
print(model.summary())

In [ ]:
wt = 1 / smf.ols('model.resid.abs() ~ model.fittedvalues', data=penguins).fit().fittedvalues**2

model2 = smf.wls("body_mass_g ~ flipper_length_mm", data=penguins, weights=wt).fit() #
print(model2.summary())

For WLS, we defined weights by getting OLS residuals first. This tells us how the variance or error is like for each observation, and we can adjust our model from there. It did a bit better if looking at the R-Squared metric!

## Assignment Section

**Question 1.** Using the `titanic` dataset from seaborn, fit a binomial GLM predicting survived from fare, age, and sex. Drop missing values first. Store your result in glm_titanic.

Note: above, I used the regulalr `sm` and not the `smf` that we are used to, which uses R formulations. Because `sex` is categorical, it will be MUCH easier to use `smf`, so use that! You will need to look up the documentation, but the format will be the same as the `WLS` example with the additional `family` parameter.

In [ ]:
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

titanic = ...

glm_titanic = ...
print(glm_titanic.summary())

In [ ]:
grader.check("q1")

**Question 2.**

Using the `mpg` dataset from seaborn, fit an OLS model predicting mpg from horsepower. Then fit a WLS model using the same formula with weights defined as 1 / fitted_values**2 where fitted_values comes from your OLS model. Store your WLS result in wls_mpg.


In [ ]:
import seaborn as sns
import statsmodels.formula.api as smf
import pandas as pd

mpg = sns.load_dataset('mpg').dropna(subset=['mpg', 'horsepower'])
mpg['horsepower'] = pd.to_numeric(mpg['horsepower'], errors='coerce')
mpg = mpg.dropna(subset=['horsepower'])

ols_mpg = ...

weights = ...

wls_mpg = ...
print(wls_mpg.summary())

In [ ]:
grader.check("q2")

**Question 3.**
Which of the following best describes why we use the absolute value of OLS residuals to construct WLS weights (as in 1 / ols('resid.abs() ~ fittedvalues').fit().fittedvalues**2)?

A) Because negative residuals are errors and should be excluded from the model

B) Because we want to model how the magnitude of error changes with fitted values, regardless of direction

C) Because (only) squaring the residuals would cause the weights to be too large

Return the answer as a string to `ans`. For example:

`ans = 'A'`

Note: Google is your friend!

In [ ]:
ans = ...

In [ ]:
grader.check("q3")

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()